<a href="https://colab.research.google.com/github/Sehar-Gillani/TCGA-RNA-seq-Differential-Expression-Analysis-Tutorial---Biomni.Phylo/blob/main/TCGA_Tutorial_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 TCGA RNA-seq Differential Expression Analysis Tutorial

## Learn Python + Bioinformatics Together!

**What you'll learn:**
- Python basics (variables, functions, data structures)
- Working with real cancer genomics data (TCGA)
- Differential expression analysis with DESeq2
- Data visualization (volcano plots, heatmaps)
- Biological interpretation of results

**Prerequisites:** None! We'll explain everything from scratch.

**Time:** ~2-3 hours

---

## 📚 Table of Contents

1. [Setup & Installation](#setup)
2. [Python Basics Crash Course](#python-basics)
3. [Understanding the Biology](#biology)
4. [Downloading TCGA Data](#download-data)
5. [Data Exploration](#explore-data)
6. [Quality Control & Preprocessing](#qc)
7. [Differential Expression Analysis](#de-analysis)
8. [Visualization](#visualization)
9. [Biological Interpretation](#interpretation)
10. [Practice Exercises](#exercises)

---
<a name="setup"></a>
# 1. 🔧 Setup & Installation

First, we need to install the required packages. In Google Colab, we use `!pip install` to install Python packages.

In [1]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
# The '!' at the start means "run this as a terminal command"
# pip is Python's package installer

!pip install pydeseq2 adjustText gseapy --quiet

# --quiet means don't show all the installation details
print("✅ Packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.0/596.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 3.1 MB/s eta 0:00:00
✅ Packages installed successfully!


In [6]:
# ============================================================
# CELL 2: Import libraries
# ============================================================
# 'import' loads a library so we can use its functions
# 'as' gives it a shorter nickname

import pandas as pd          # For working with tables (DataFrames)
import numpy as np           # For numerical operations
import matplotlib.pyplot as plt  # For creating plots
import seaborn as sns        # For beautiful statistical plots
import requests              # For downloading data from the internet
import gzip                  # For handling compressed files
import io                    # For handling data streams
import warnings
warnings.filterwarnings('ignore')  # Hide warning messages

# Set plot style
sns.set_style("ticks")
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries imported!")

✅ All libraries imported!


---
<a name="python-basics"></a>
# 2. 🐍 Python Basics Crash Course

Before we dive into the analysis, let's learn some Python fundamentals.

### 2.1 Variables
Variables store data. Think of them as labeled boxes.

In [8]:
# ============================================================
# CELL 3: Variables - storing data
# ============================================================

# A variable is like a labeled box that holds data
# The '=' sign means "assign this value to this variable"

gene_name = "TP53"           # A string (text) - use quotes
expression_level = 1523.5    # A number (float - has decimals)
sample_count = 100           # A number (integer - whole number)
is_significant = True        # A boolean (True or False)

# Print the values
print("Gene name:", gene_name)
print("Expression level:", expression_level)
print("Sample count:", sample_count)
print("Is significant?", is_significant)

Gene name: TP53
Expression level: 1523.5
Sample count: 100
Is significant? True


In [10]:
# ============================================================
# CELL 4: Lists - storing multiple items
# ============================================================

# A list holds multiple items in order
# Use square brackets []

genes = ["TP53", "BRCA1", "EGFR", "MYC"]
expression_values = [1523.5, 892.3, 2341.1, 567.8]

# Access items by index (position) - Python counts from 0!
print("First gene:", genes[0])      # TP53
print("Second gene:", genes[1])     # BRCA1
print("Last gene:", genes[-1])      # MYC (negative = from end)

# How many items?
print("Number of genes:", len(genes))

First gene: TP53
Second gene: BRCA1
Last gene: MYC
Number of genes: 4


In [4]:
# ============================================================
# CELL 5: Dictionaries - key-value pairs
# ============================================================

# A dictionary maps keys to values (like a real dictionary!)
# Use curly braces {}

gene_info = {
    "name": "TP53",
    "chromosome": 17,
    "function": "tumor suppressor",
    "expression": 1523.5
}

# Access values by key
print("Gene name:", gene_info["name"])
print("Function:", gene_info["function"])

# Add a new key-value pair
gene_info["is_oncogene"] = False
print("Updated info:", gene_info)

Gene name: TP53
Function: tumor suppressor
Updated info: {'name': 'TP53', 'chromosome': 17, 'function': 'tumor suppressor', 'expression': 1523.5, 'is_oncogene': False}


In [11]:
# ============================================================
# CELL 6: DataFrames - tables of data (MOST IMPORTANT!)
# ============================================================

# A DataFrame is like an Excel spreadsheet
# It's the main data structure we'll use for analysis

# Create a simple DataFrame
data = {
    "gene": ["TP53", "BRCA1", "EGFR", "MYC"],
    "tumor_expression": [1523, 892, 2341, 567],
    "normal_expression": [1200, 950, 1100, 200]
}

df = pd.DataFrame(data)  # pd is our nickname for pandas
print(df)
print()

# Useful DataFrame operations:
print("Shape (rows, columns):", df.shape)
print("Column names:", df.columns.tolist())
print("First 2 rows:")
print(df.head(2))

    gene  tumor_expression  normal_expression
0   TP53              1523               1200
1  BRCA1               892                950
2   EGFR              2341               1100
3    MYC               567                200

Shape (rows, columns): (4, 3)
Column names: ['gene', 'tumor_expression', 'normal_expression']
First 2 rows:
    gene  tumor_expression  normal_expression
0   TP53              1523               1200
1  BRCA1               892                950


In [ ]:
# ============================================================
# CELL 7: DataFrame operations
# ============================================================

# Select a single column (returns a Series)
print("Tumor expression values:")
print(df["tumor_expression"])
print()

# Calculate a new column
df["fold_change"] = df["tumor_expression"] / df["normal_expression"]
print("With fold change:")
print(df)
print()

# Filter rows based on a condition
upregulated = df[df["fold_change"] > 1.5]
print("Upregulated genes (FC > 1.5):")
print(upregulated)

### 🎯 Quick Exercise 1

Try modifying the code above to:
1. Add a new gene "KRAS" with tumor_expression=800 and normal_expression=400
2. Filter for genes with fold_change > 2

*(Hint: Use `df.loc[len(df)] = ["KRAS", 800, 400, 2.0]` to add a row)*

---
<a name="biology"></a>
# 3. 🧬 Understanding the Biology

Before we analyze data, let's understand what we're doing and why.

## What is Differential Expression Analysis?

**Goal:** Find genes that are turned ON or OFF differently between two conditions.

```
Example: Tumor vs Normal tissue

Gene X:  Tumor = 5000 counts    Normal = 500 counts   → UPREGULATED in tumor
Gene Y:  Tumor = 200 counts     Normal = 2000 counts  → DOWNREGULATED in tumor
Gene Z:  Tumor = 1000 counts    Normal = 1050 counts  → NO CHANGE
```

## Key Concepts

### 1. RNA-seq Counts
- RNA-seq measures gene expression by counting RNA molecules
- Higher count = gene is more active (more RNA being made)
- We get ~20,000 genes × N samples

### 2. Log2 Fold Change (log2FC)
- Measures HOW MUCH expression changes
- `log2FC = log2(Tumor / Normal)`
- log2FC = +1 means 2x higher in tumor
- log2FC = -1 means 2x lower in tumor
- log2FC = 0 means no change

### 3. Adjusted P-value (padj)
- Measures statistical significance
- padj < 0.05 = statistically significant
- We test 20,000 genes, so we must correct for multiple testing!

### 4. What is TCGA?
- **T**he **C**ancer **G**enome **A**tlas
- Largest cancer genomics project
- 33 cancer types, 11,000+ patients
- Free, public data!

---
<a name="download-data"></a>
# 4. 📥 Downloading TCGA Data

We'll download real breast cancer data from TCGA.

In [ ]:
# ============================================================
# CELL 8: Download TCGA-BRCA expression data
# ============================================================

# We'll download from UCSC Xena - a user-friendly TCGA data portal
# This is log2(count+1) normalized data

print("Downloading TCGA-BRCA expression data...")
print("Source: UCSC Xena Browser (https://xenabrowser.net)")
print("This may take 1-2 minutes...")
print()

# URL for TCGA-BRCA RNA-seq data
url = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHiSeqV2.gz"

# Download and decompress
response = requests.get(url, timeout=120)

if response.status_code == 200:
    # Decompress the gzipped file
    with gzip.open(io.BytesIO(response.content), 'rt') as f:
        expr_data = pd.read_csv(f, sep='\t', index_col=0)

    print(f"✅ Downloaded successfully!")
    print(f"   Shape: {expr_data.shape[0]} genes × {expr_data.shape[1]} samples")
else:
    print(f"❌ Download failed with status code: {response.status_code}")

In [ ]:
# ============================================================
# CELL 9: Understand TCGA sample IDs
# ============================================================

# TCGA sample IDs encode important information!
# Format: TCGA-XX-XXXX-XXY
#         └─┬─┘ └──┬──┘ └┬┘
#           │     │     └── Sample type (01-09=Tumor, 10-19=Normal)
#           │     └──────── Patient ID
#           └────────────── Project code

print("Example sample IDs:")
print(expr_data.columns[:5].tolist())
print()

# Let's decode them
def get_sample_type(sample_id):
    """
    Extract sample type from TCGA barcode.
    Sample codes 01-09 = Tumor
    Sample codes 10-19 = Normal
    """
    try:
        parts = sample_id.split('-')
        if len(parts) >= 4:
            sample_code = int(parts[3][:2])
            if 1 <= sample_code <= 9:
                return 'Tumor'
            elif 10 <= sample_code <= 19:
                return 'Normal'
        return 'Unknown'
    except:
        return 'Unknown'

# Test the function
test_samples = ['TCGA-A7-A0CH-01', 'TCGA-A7-A0CH-11']
for s in test_samples:
    print(f"{s} → {get_sample_type(s)}")

# Note: -01 = Tumor, -11 = Normal

In [ ]:
# ============================================================
# CELL 10: Create sample metadata
# ============================================================

# Create a metadata table with sample information
sample_ids = expr_data.columns.tolist()

metadata = pd.DataFrame({
    'sample_id': sample_ids,
    'condition': [get_sample_type(s) for s in sample_ids]
})

# Count samples by type
print("Sample distribution:")
print(metadata['condition'].value_counts())
print()

# Keep only Tumor and Normal samples
metadata = metadata[metadata['condition'].isin(['Tumor', 'Normal'])]
print(f"Keeping {len(metadata)} samples (Tumor + Normal)")

In [ ]:
# ============================================================
# CELL 11: Select a balanced subset for faster analysis
# ============================================================

# For this tutorial, we'll use a smaller subset
# (Full analysis would take longer)

np.random.seed(42)  # For reproducibility - same "random" results each time

# Get sample IDs by condition
tumor_samples = metadata[metadata['condition'] == 'Tumor']['sample_id'].tolist()
normal_samples = metadata[metadata['condition'] == 'Normal']['sample_id'].tolist()

print(f"Available: {len(tumor_samples)} tumor, {len(normal_samples)} normal")

# Select 50 of each (or all normal if fewer than 50)
n_samples = min(50, len(normal_samples))
selected_tumor = np.random.choice(tumor_samples, size=n_samples, replace=False).tolist()
selected_normal = normal_samples[:n_samples]

selected_samples = selected_tumor + selected_normal
print(f"Selected: {len(selected_tumor)} tumor + {len(selected_normal)} normal = {len(selected_samples)} total")

# Subset the data
expr_subset = expr_data[selected_samples].copy()

# Create metadata for subset
metadata_subset = pd.DataFrame({
    'sample_id': selected_samples,
    'condition': ['Tumor'] * len(selected_tumor) + ['Normal'] * len(selected_normal)
})
metadata_subset = metadata_subset.set_index('sample_id')

print(f"\nFinal data shape: {expr_subset.shape[0]} genes × {expr_subset.shape[1]} samples")

In [ ]:
# ============================================================
# CELL 12: Convert to raw counts for DESeq2
# ============================================================

# The downloaded data is log2(count+1) transformed
# DESeq2 needs RAW COUNTS (integers)
# So we reverse the transformation: count = 2^(value) - 1

print("Converting log2(count+1) to raw counts...")
print()

# Show before
print("Before (log2 transformed):")
print(expr_subset.iloc[:3, :3].round(2))
print()

# Convert
counts = (2 ** expr_subset) - 1
counts = counts.round().astype(int)  # DESeq2 needs integers

# Show after
print("After (raw counts):")
print(counts.iloc[:3, :3])
print()

print("✅ Data ready for analysis!")

---
<a name="explore-data"></a>
# 5. 🔍 Data Exploration

Before analysis, always explore your data!

In [ ]:
# ============================================================
# CELL 13: Basic data exploration
# ============================================================

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)

print(f"\n📊 Expression Matrix:")
print(f"   Genes (rows): {counts.shape[0]:,}")
print(f"   Samples (columns): {counts.shape[1]}")

print(f"\n📋 Sample Groups:")
print(metadata_subset['condition'].value_counts().to_string())

print(f"\n📈 Expression Statistics:")
print(f"   Min count: {counts.values.min():,}")
print(f"   Max count: {counts.values.max():,}")
print(f"   Mean count: {counts.values.mean():,.1f}")

In [ ]:
# ============================================================
# CELL 14: Visualize sample distribution
# ============================================================

# Plot total counts per sample (library size)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Calculate total counts per sample
total_counts = counts.sum(axis=0)

# Add condition info
plot_data = pd.DataFrame({
    'total_counts': total_counts,
    'condition': metadata_subset.loc[total_counts.index, 'condition']
})

# Plot 1: Boxplot by condition
sns.boxplot(data=plot_data, x='condition', y='total_counts', ax=axes[0])
axes[0].set_title('Total Counts by Condition')
axes[0].set_ylabel('Total Counts (millions)')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(6,6))

# Plot 2: Histogram
axes[1].hist(total_counts / 1e6, bins=20, edgecolor='black')
axes[1].set_xlabel('Total Counts (millions)')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Distribution of Library Sizes')

plt.tight_layout()
plt.show()

print("💡 Similar library sizes across conditions = good!")

---
<a name="qc"></a>
# 6. ✅ Quality Control & Preprocessing

Critical step! Check for issues before analysis.

In [ ]:
# ============================================================
# CELL 15: Quality control checks
# ============================================================

print("=" * 60)
print("QUALITY CONTROL CHECKS")
print("=" * 60)

# Check 1: Missing values
missing = counts.isnull().sum().sum()
print(f"\n1. Missing values: {missing}")
if missing == 0:
    print("   ✅ No missing values!")
else:
    print("   ⚠️ Missing values detected - need to handle")

# Check 2: Duplicate genes
dup_genes = counts.index.duplicated().sum()
print(f"\n2. Duplicate genes: {dup_genes}")
if dup_genes == 0:
    print("   ✅ No duplicate genes!")
else:
    print("   ⚠️ Duplicates found - removing...")
    counts = counts[~counts.index.duplicated(keep='first')]

# Check 3: Sample ID alignment
data_samples = set(counts.columns)
meta_samples = set(metadata_subset.index)
aligned = data_samples == meta_samples
print(f"\n3. Sample IDs aligned: {aligned}")
if aligned:
    print("   ✅ All sample IDs match!")
else:
    print("   ⚠️ Mismatch detected - fixing...")
    common = data_samples & meta_samples
    counts = counts[list(common)]
    metadata_subset = metadata_subset.loc[list(common)]

In [ ]:
# ============================================================
# CELL 16: Filter lowly expressed genes
# ============================================================

# WHY FILTER?
# - Genes with very low counts are mostly noise
# - They have unreliable statistics
# - Removing them reduces multiple testing burden

print("Filtering lowly expressed genes...")
print()

# Rule: Keep genes with at least 10 counts in at least N samples
# where N = size of smallest group
min_counts = 10
min_samples = min(metadata_subset['condition'].value_counts())

print(f"Criteria: ≥{min_counts} counts in ≥{min_samples} samples")
print(f"Genes before filtering: {counts.shape[0]:,}")

# Count how many samples have >= min_counts for each gene
samples_above_threshold = (counts >= min_counts).sum(axis=1)
genes_to_keep = samples_above_threshold >= min_samples

counts_filtered = counts[genes_to_keep]

print(f"Genes after filtering: {counts_filtered.shape[0]:,}")
print(f"Genes removed: {counts.shape[0] - counts_filtered.shape[0]:,}")
print()
print("✅ Data preprocessed and ready for DESeq2!")

---
<a name="de-analysis"></a>
# 7. 📊 Differential Expression Analysis

Now the main analysis using DESeq2!

In [ ]:
# ============================================================
# CELL 17: Run DESeq2 analysis
# ============================================================

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print("=" * 60)
print("RUNNING DESeq2 ANALYSIS")
print("=" * 60)

# DESeq2 expects: samples as ROWS, genes as COLUMNS
# Our data has: genes as ROWS, samples as COLUMNS
# So we need to TRANSPOSE (flip rows and columns)

counts_for_deseq = counts_filtered.T  # .T means transpose
print(f"\nData shape for DESeq2: {counts_for_deseq.shape[0]} samples × {counts_for_deseq.shape[1]} genes")

# Create DESeq2 dataset
print("\nCreating DESeq2 dataset...")
print("  Reference condition: Normal")
print("  Comparison: Tumor vs Normal")
print("  (Positive log2FC = higher in Tumor)")

dds = DeseqDataSet(
    counts=counts_for_deseq,
    metadata=metadata_subset,
    design_factors="condition",
    n_cpus=2
)

# Run the analysis
print("\nRunning DESeq2 (this may take a few minutes)...")
dds.deseq2()
print("✅ DESeq2 analysis complete!")

In [ ]:
# ============================================================
# CELL 18: Extract and understand results
# ============================================================

# Run statistical tests
stat_res = DeseqStats(dds, contrast=["condition", "Tumor", "Normal"], n_cpus=2)
stat_res.summary()

# Get results as DataFrame
results = stat_res.results_df.copy()

print("\n" + "=" * 60)
print("UNDERSTANDING THE RESULTS")
print("=" * 60)

print(f"\nTested {len(results):,} genes")
print("\nColumn meanings:")
print("  • baseMean: Average expression across all samples")
print("  • log2FoldChange: log2(Tumor/Normal) - positive = higher in tumor")
print("  • lfcSE: Standard error of log2FC")
print("  • stat: Wald test statistic")
print("  • pvalue: Raw p-value (DON'T use for filtering!)")
print("  • padj: Adjusted p-value (USE THIS for filtering!)")

print("\nFirst 10 results:")
print(results.head(10).round(4))

In [ ]:
# ============================================================
# CELL 19: Filter significant genes
# ============================================================

print("=" * 60)
print("FILTERING SIGNIFICANT GENES")
print("=" * 60)

# Standard thresholds:
# - padj <= 0.05 (statistically significant)
# - |log2FC| >= 0.5 (biologically meaningful, ~1.4-fold change)

padj_threshold = 0.05
log2fc_threshold = 0.5

print(f"\nThresholds:")
print(f"  • padj ≤ {padj_threshold}")
print(f"  • |log2FC| ≥ {log2fc_threshold}")

# Filter for significant DEGs
# IMPORTANT: Use <= for padj (inclusive)
significant = results[
    (results['padj'] <= padj_threshold) &
    (abs(results['log2FoldChange']) >= log2fc_threshold)
].copy()

# Separate up and down regulated
upregulated = significant[significant['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)
downregulated = significant[significant['log2FoldChange'] < 0].sort_values('log2FoldChange')

print(f"\n📊 RESULTS:")
print(f"   Total genes tested: {len(results):,}")
print(f"   Significant DEGs: {len(significant):,}")
print(f"   ↑ Up-regulated in Tumor: {len(upregulated):,}")
print(f"   ↓ Down-regulated in Tumor: {len(downregulated):,}")

In [ ]:
# ============================================================
# CELL 20: View top genes
# ============================================================

print("🔺 TOP 10 UP-REGULATED GENES (higher in Tumor):")
print("=" * 60)
print(upregulated[['baseMean', 'log2FoldChange', 'padj']].head(10).round(4))

print("\n🔻 TOP 10 DOWN-REGULATED GENES (lower in Tumor):")
print("=" * 60)
print(downregulated[['baseMean', 'log2FoldChange', 'padj']].head(10).round(4))

---
<a name="visualization"></a>
# 8. 📈 Visualization

Create publication-quality figures!

In [ ]:
# ============================================================
# CELL 21: Create Volcano Plot
# ============================================================

from adjustText import adjust_text

# Prepare data
plot_df = results.copy()
plot_df['-log10_padj'] = -np.log10(plot_df['padj'].clip(lower=1e-300))

# Categorize genes
def categorize(row):
    if pd.isna(row['padj']):
        return 'Not Significant'
    elif row['padj'] <= 0.05 and row['log2FoldChange'] >= 0.5:
        return 'Up-regulated'
    elif row['padj'] <= 0.05 and row['log2FoldChange'] <= -0.5:
        return 'Down-regulated'
    else:
        return 'Not Significant'

plot_df['category'] = plot_df.apply(categorize, axis=1)

# Create plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = {'Not Significant': '#CCCCCC', 'Up-regulated': '#E74C3C', 'Down-regulated': '#3498DB'}

for category in ['Not Significant', 'Down-regulated', 'Up-regulated']:
    subset = plot_df[plot_df['category'] == category]
    ax.scatter(subset['log2FoldChange'], subset['-log10_padj'],
               c=colors[category], label=f'{category} ({len(subset)})',
               alpha=0.6, s=10)

# Add threshold lines
ax.axhline(y=-np.log10(0.05), color='gray', linestyle='--', linewidth=1)
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1)
ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=1)

# Label top genes
top_genes = pd.concat([
    plot_df[plot_df['category'] == 'Up-regulated'].nlargest(5, '-log10_padj'),
    plot_df[plot_df['category'] == 'Down-regulated'].nlargest(5, '-log10_padj')
])

texts = []
for idx, row in top_genes.iterrows():
    texts.append(ax.annotate(idx, (row['log2FoldChange'], row['-log10_padj']), fontsize=8))
adjust_text(texts)

ax.set_xlabel('log2 Fold Change (Tumor vs Normal)', fontsize=12)
ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
ax.set_title('Volcano Plot: Breast Cancer Differential Expression', fontsize=14)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('volcano_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: volcano_plot.png")

In [ ]:
# ============================================================
# CELL 22: Create Heatmap of Top DEGs
# ============================================================

# Select top 20 up and 20 down regulated genes
top_up = upregulated.head(20).index.tolist()
top_down = downregulated.head(20).index.tolist()
top_genes_list = top_up + top_down

# Get expression data for these genes
heatmap_data = counts_filtered.loc[top_genes_list]

# Log transform and z-score normalize
heatmap_log = np.log2(heatmap_data + 1)
heatmap_zscore = (heatmap_log.T - heatmap_log.mean(axis=1)) / heatmap_log.std(axis=1)
heatmap_zscore = heatmap_zscore.T

# Sort samples by condition
sample_order = metadata_subset.sort_values('condition').index.tolist()
heatmap_zscore = heatmap_zscore[sample_order]

# Create color annotations
condition_colors = metadata_subset.loc[sample_order, 'condition'].map(
    {'Tumor': '#E74C3C', 'Normal': '#3498DB'}
)

# Create heatmap
g = sns.clustermap(heatmap_zscore,
                   cmap='RdBu_r',
                   center=0,
                   col_cluster=False,
                   row_cluster=True,
                   col_colors=condition_colors,
                   yticklabels=True,
                   xticklabels=False,
                   figsize=(12, 10),
                   cbar_kws={'label': 'Z-score'})

g.fig.suptitle('Top 40 Differentially Expressed Genes', y=1.02, fontsize=14)

plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: heatmap.png")

---
<a name="interpretation"></a>
# 9. 🧠 Biological Interpretation

What do these genes mean biologically?

In [ ]:
# ============================================================
# CELL 23: Save results
# ============================================================

# Save all results
results.to_csv('deseq2_results_all.csv')
significant.to_csv('significant_degs.csv')
upregulated.to_csv('upregulated_genes.csv')
downregulated.to_csv('downregulated_genes.csv')

print("✅ Results saved:")
print("   • deseq2_results_all.csv - All genes")
print("   • significant_degs.csv - Significant DEGs only")
print("   • upregulated_genes.csv - Up-regulated genes")
print("   • downregulated_genes.csv - Down-regulated genes")
print()
print("📥 Download these files from the Files panel on the left!")
print("   (Click the folder icon → right-click file → Download)")

In [ ]:
# ============================================================
# CELL 24: Summary of findings
# ============================================================

print("=" * 70)
print("                    ANALYSIS SUMMARY")
print("=" * 70)

print(f"\n📊 DATA:")
print(f"   • Source: TCGA-BRCA (Breast Cancer)")
print(f"   • Samples: {len(metadata_subset)} ({(metadata_subset['condition']=='Tumor').sum()} Tumor, {(metadata_subset['condition']=='Normal').sum()} Normal)")
print(f"   • Genes tested: {len(results):,}")

print(f"\n📈 RESULTS:")
print(f"   • Significant DEGs: {len(significant):,}")
print(f"   • Up-regulated in Tumor: {len(upregulated):,}")
print(f"   • Down-regulated in Tumor: {len(downregulated):,}")

print(f"\n🔺 TOP UP-REGULATED GENES:")
for gene in upregulated.head(5).index:
    fc = upregulated.loc[gene, 'log2FoldChange']
    print(f"   • {gene}: {fc:.2f} log2FC ({2**fc:.1f}x higher in tumor)")

print(f"\n🔻 TOP DOWN-REGULATED GENES:")
for gene in downregulated.head(5).index:
    fc = downregulated.loc[gene, 'log2FoldChange']
    print(f"   • {gene}: {fc:.2f} log2FC ({2**abs(fc):.1f}x lower in tumor)")

---
<a name="exercises"></a>
# 10. 🎯 Practice Exercises

Try these exercises to reinforce your learning!

## Exercise 1: Change the thresholds

Modify the filtering code to use stricter thresholds:
- padj ≤ 0.01 (instead of 0.05)
- |log2FC| ≥ 1.0 (instead of 0.5)

How many significant genes do you get now?

In [ ]:
# YOUR CODE HERE
# Hint: Copy the filtering code from Cell 19 and change the thresholds

padj_threshold = ___  # Fill in
log2fc_threshold = ___  # Fill in

significant_strict = results[
    (results['padj'] <= padj_threshold) &
    (abs(results['log2FoldChange']) >= log2fc_threshold)
]

print(f"Significant genes with strict thresholds: {len(significant_strict)}")

## Exercise 2: Find a specific gene

Look up the results for TP53 (a famous tumor suppressor gene).
Is it differentially expressed in breast cancer?

In [ ]:
# YOUR CODE HERE
# Hint: Use results.loc['GENE_NAME'] to look up a specific gene

gene_to_find = "TP53"

if gene_to_find in results.index:
    print(results.loc[gene_to_find])
else:
    print(f"{gene_to_find} not found in results")

## Exercise 3: Analyze a different cancer type

Try changing the cancer type! Replace `TCGA.BRCA` with one of these:
- `TCGA.LUAD` (Lung Adenocarcinoma)
- `TCGA.COAD` (Colon Adenocarcinoma)
- `TCGA.PRAD` (Prostate Adenocarcinoma)

Go back to Cell 8 and change the URL, then re-run all cells!

---
# 🎉 Congratulations!

You've completed a full differential expression analysis!

## What you learned:
- Python basics (variables, lists, DataFrames)
- How to download and process TCGA data
- Quality control and preprocessing
- Running DESeq2 for differential expression
- Creating volcano plots and heatmaps
- Interpreting biological results

## Next steps:
1. **Pathway analysis** - Find which biological pathways are affected
2. **Gene Ontology enrichment** - What functions are enriched?
3. **Survival analysis** - Do these genes predict patient outcomes?
4. **Apply to your own data!**

## Resources:
- [TCGA Data Portal](https://portal.gdc.cancer.gov/)
- [DESeq2 Documentation](https://bioconductor.org/packages/release/bioc/html/DESeq2.html)
- [Python for Biologists](https://pythonforbiologists.com/)